#### Adversarial Search Lab Practice

Minmax algorithm

In [87]:
UTILITY = {
    'A1': 3, 'A2': 5,
    'B1': 2, 'B2': 9,
    'C1': 0, 'C2': 7,
    'D1': 4, 'D2': 6,
    'E1': 1, 'E2': 8,
    'F1': 5, 'F2': 2,
    'G1': 6, 'G2': 4,
    'H1': 7, 'H2': 3
}

GAME_TREE = {
    'R': ['A', 'B'],
    'A': ['C', 'D'],
    'B': ['E', 'F'],
    'C': ['A1', 'A2'],
    'D': ['B1', 'B2'],
    'E': ['C1', 'C2'],
    'F': ['D1', 'D2'],
    'A1': [], 'A2': [],
    'B1': [], 'B2': [],
    'C1': [], 'C2': [],
    'D1': [], 'D2': []
}

# UTILITY = {
#     'L1': 3, 'L2': 5, 'L3': 2, 'L4': 9,
#     'L5': 0, 'L6': 7, 'L7': 4, 'L8': 6,
#     'L9': 1, 'L10': 8, 'L11': 5, 'L12': 2,
#     'L13': 6, 'L14': 4, 'L15': 7, 'L16': 3
# }

# GAME_TREE = {
#     'R': ['A', 'B'],
#     'A': ['C', 'D'],
#     'B': ['E', 'F'],
#     'C': ['G', 'H'],
#     'D': ['I', 'J'],
#     'E': ['K', 'L'],
#     'F': ['M', 'N'],
#     'G': ['L1', 'L2'], 'H': ['L3', 'L4'],
#     'I': ['L5', 'L6'], 'J': ['L7', 'L8'],
#     'K': ['L9', 'L10'], 'L': ['L11', 'L12'],
#     'M': ['L13', 'L14'], 'N': ['L15', 'L16']
# }

In [44]:
def minimax(node, maximizing_player=True):
    if node in UTILITY:
        return UTILITY[node]

    children = GAME_TREE[node]

    if maximizing_player:
        return max(minimax(child, maximizing_player=False) for child in children)
    else :
        return min(minimax(child, maximizing_player=True) for child in children)

import math
def minimax_decision(root='R'):
    best_move = None
    best_value = -math.inf

    children = GAME_TREE[root]
    for child in children:
        value = minimax(child, maximizing_player=False)
        if value > best_value:
            best_value = value
            best_move = child

    return best_value, best_move

In [45]:
value, move = minimax_decision()
print(value, move)

3 A


In [88]:
# alpha beta pruning, which returns value and stats {prune, leaves_visited}
def alpha_beta(node, alpha=-math.inf, beta=math.inf, maximizing_player=True, stats=None):
    if stats == None:
        stats = {'prune': 0, 'leaf': 0}

    if node in UTILITY:
        stats['leaf'] += 1
        return UTILITY[node], stats

    children = GAME_TREE[node]

    if maximizing_player:
        best_move = None
        best_value = -math.inf
        for child in children:
            value, stats = alpha_beta(child, alpha, beta, False, stats)
            if value > best_value:
                best_value = value
                best_move = child
            alpha = max(alpha, value)
            if alpha >= beta:
                stats['prune'] += 1
                break
        return best_value, stats
            
    # else:
    best_move = None
    best_value = math.inf
    for child in children:
        
        value, stats = alpha_beta(child, alpha, beta, True, stats)
        # print(value)
        if value < best_value:
            best_value = value
            best_move = child
        beta = min(beta, value)
        if alpha >= beta:
            stats['prune'] += 1
            break
    return best_value, stats 


def alpha_beta_decision(root='R'):
    
    best_move = None
    best_value = -math.inf
    alpha = -math.inf
    beta = math.inf
    stats = {'prune': 0, 'leaf': 0}

    children = GAME_TREE[root]
    for child in children:
        value, stats = alpha_beta(child, alpha, beta, False, stats)

        if value > best_value:
            best_value = value
            best_move = child
        alpha = max(alpha, value)
        if alpha >= beta:
            stats['prune'] += 1
            break
    return best_value, stats

In [89]:
value, stats = alpha_beta_decision()
print(value, stats)

6 {'prune': 1, 'leaf': 8}


In [77]:
def nim_successors(state):
    successor = set()

    for pile, stones in enumerate(state):
        for remaining in range(stones):
            new_state = list(state)

            if remaining == 0:
                new_state.pop(pile)
            else :
                new_state[pile] = remaining

            new_state = tuple(sorted((x for x in new_state if x > 0), reverse=True ))
            successor.add(new_state)

    return sorted(list(successor), reverse=True)



def nim_is_winning(state):

    successor = nim_successors(state)

    if not successor:
        return False

    return any(not nim_is_winning(next_state) for next_state in successor)

def best_move(state):
    print([next_state for next_state in nim_successors(state) 
            if not nim_is_winning(next_state)])
    return [next_state for next_state in nim_successors(state) 
            if not nim_is_winning(next_state)]


In [ ]:
# nim_successors((3, 4))


0 3 <class 'int'>
1 4 <class 'int'>


[(4, 2), (4, 1), (4,), (3, 3), (3, 2), (3, 1), (3,)]

In [81]:
check = nim_is_winning((3, 2))
move = best_move((3, 2))
print(check, move)

[(2, 2)]
True [(2, 2)]


In [67]:
new_state = [3, 2, 0, 1, -4 , 5]
l1 = tuple(sorted((x for x in new_state if x > 0), reverse=True ))
# l1 = [x for x in new_state  if x> 0]
l1

suc = set()

suc.add(l1)
print(suc)

{(5, 3, 2, 1)}


In [83]:
from functools import lru_cache

@lru_cache(maxsize=None)
def grundy_successor(state):
    successor = set()

    for i, n in enumerate(state):
        for a in range(1, n):
            b = n - a

            if a == b:
                continue

            new_state = list(state)
            new_state.pop(i)
            new_state.extend([a, b])

            new_state = tuple(sorted(new_state, reverse=True))
            successor.add(new_state)

    return tuple(sorted(successor, reverse=True))


@lru_cache(maxsize=None)
def grundy_is_winning(state):
    successor = grundy_successor(state)

    if not successor:
        return False

    return any(not grundy_successor(next_state) for next_state in successor)

def grundy_best_move(state):

    return tuple(next_state for next_state in grundy_successor(state)
                 if not grundy_is_winning(next_state))




In [85]:
initial_grundy_state = (5,)

grundy_best_move(initial_grundy_state)

((4, 1),)

In [86]:
grundy_is_winning(initial_grundy_state)

False

In [ ]:
# input as adj list

n = int(input())

GAME_TREE = {}

for _ in range(n):
    data = input().split()
    node = data[0]
    children = data[1:]

    if children == ['-']:
        children = []

    GAME_TREE[node] = children

# Read utilities in the order leaves appear in GAME_TREE
leaves = [node for node, children in GAME_TREE.items() if not children]

values = list(map(int, input().split()))

if len(values) != len(leaves):
    raise ValueError("Number of utilities must equal number of leaves")

UTILITY = dict(zip(leaves, values))

print("GAME_TREE =", GAME_TREE)
print("UTILITY =", UTILITY)

In [ ]:
# input as edge list
import math

# ---------- Edge List Input ----------
n = int(input("Number of edges: "))

GAME_TREE = {}

for _ in range(n):
    parent, child = input().split()

    if parent not in GAME_TREE:
        GAME_TREE[parent] = []

    GAME_TREE[parent].append(child)

    # Ensure every child is also a node
    if child not in GAME_TREE:
        GAME_TREE[child] = []

# ---------- Utility Input ----------
leaves = [
    node for node, children in GAME_TREE.items()
    if len(children) == 0
]

print("Leaf nodes:", leaves)
print("Enter utility values in the above order:")

values = list(map(int, input().split()))

if len(values) != len(leaves):
    raise ValueError("Number of utilities must equal number of leaves")

UTILITY = dict(zip(leaves, values))

print("GAME_TREE =", GAME_TREE)
print("UTILITY =", UTILITY)

In [ ]:
import sys, math
from collections import defaultdict


def run_tests(solver, tests):
    """solver: function str -> str; tests: list of (input_text, expected_text)"""
    passed = 0

    for i, (inp, expected) in enumerate(tests, 1):
        got = solver(inp)

        ok = [l.strip() for l in got.strip().splitlines()] == \
             [l.strip() for l in expected.strip().splitlines()]

        passed += ok

        print(f"Test {i}: {'PASS' if ok else 'FAIL'}")

        if not ok:
            print("--- expected ---\n" + expected.strip())
            print("--- got ---\n" + got.strip())

    print(f"==> {passed}/{len(tests)} test cases passed\n")


def parse_input(data):
    tokens = data.split()
    pos = 0

    # Number of edges
    m = int(tokens[pos])
    pos += 1

    GAME_TREE = {}

    for _ in range(m):
        parent = tokens[pos]
        child = tokens[pos + 1]
        pos += 2

        if parent not in GAME_TREE:
            GAME_TREE[parent] = []

        GAME_TREE[parent].append(child)

        if child not in GAME_TREE:
            GAME_TREE[child] = []

    # Root node
    root = tokens[pos]
    pos += 1

    # Find leaves in insertion order
    leaves = [
        node for node, children in GAME_TREE.items()
        if len(children) == 0
    ]

    # Read utility values in leaf order
    values = list(map(int, tokens[pos:]))

    if len(values) != len(leaves):
        raise ValueError("Utility count does not match leaf count")

    UTILITY = dict(zip(leaves, values))

    return GAME_TREE, UTILITY, root

In [ ]:

import math

HUMAN = 'X'
AI = 'O'
EMPTY = '-'


def print_board(board):
    for i in range(0, 9, 3):
        print(' '.join(board[i:i+3]))
    print()


def check_winner(board):
    wins = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),
        (0, 3, 6), (1, 4, 7), (2, 5, 8),
        (0, 4, 8), (2, 4, 6)
    ]

    for a, b, c in wins:
        if board[a] == board[b] == board[c] != EMPTY:
            return board[a]

    if EMPTY not in board:
        return 'DRAW'

    return None


def alpha_beta(board, alpha, beta, maximizing, depth):
    result = check_winner(board)

    if result == AI:
        return 10 - depth

    if result == HUMAN:
        return depth - 10

    if result == 'DRAW':
        return 0

    if maximizing:
        best_value = -math.inf

        for i in range(9):
            if board[i] == EMPTY:
                board[i] = AI

                value = alpha_beta(
                    board, alpha, beta, False, depth + 1
                )

                board[i] = EMPTY

                best_value = max(best_value, value)
                alpha = max(alpha, best_value)

                if alpha >= beta:
                    break

        return best_value

    else:
        best_value = math.inf

        for i in range(9):
            if board[i] == EMPTY:
                board[i] = HUMAN

                value = alpha_beta(
                    board, alpha, beta, True, depth + 1
                )

                board[i] = EMPTY

                best_value = min(best_value, value)
                beta = min(beta, best_value)

                if alpha >= beta:
                    break

        return best_value


def best_move(board):
    best_value = -math.inf
    move = -1
    alpha = -math.inf
    beta = math.inf

    for i in range(9):
        if board[i] == EMPTY:
            board[i] = AI

            value = alpha_beta(
                board, alpha, beta, False, 1
            )

            board[i] = EMPTY

            if value > best_value:
                best_value = value
                move = i

            alpha = max(alpha, best_value)

    return move


def play():
    board = [EMPTY] * 9

    print("Tic-Tac-Toe")
    print("You are X, computer is O")
    print("Positions:")
    print("0 1 2")
    print("3 4 5")
    print("6 7 8")
    print()

    while True:
        print_board(board)

        # Human move
        try:
            pos = int(input("Enter position (0-8): "))
        except ValueError:
            print("Enter a valid integer.")
            continue

        if pos < 0 or pos > 8 or board[pos] != EMPTY:
            print("Invalid move. Try again.")
            continue

        board[pos] = HUMAN

        result = check_winner(board)

        if result is not None:
            break

        # Computer move
        pos = best_move(board)
        board[pos] = AI

        print("Computer chooses:", pos)

        result = check_winner(board)

        if result is not None:
            break

    print_board(board)

    if result == 'DRAW':
        print("Game drawn!")
    else:
        print(result, "wins!")


if __name__ == "__main__":
    play()


Tic-Tac-Toe
You are X, computer is O
Positions:
0 1 2
3 4 5
6 7 8

- - -
- - -
- - -

